# Example 01: creat a simple robot model manager

In this code you will learn how to create a robot configuration file and generate a robot model manager `ManagerCasadiModel` for the robot base hunter.

___
## Model Cfg

The robot configuration class start with the attributes that defined the robot. You need ad least `name`, `nx` and `nu`. But you can add other attributes that will be accessible in the final class model as `x_label` and `u_label`.

In [ ]:
import numpy as np
import casadi as ca
from casadi import (
    vertcat,
    horzcat,
    cos,
    sin,
)
from o2r_pi2_controllers.utils import configclass, ca_euler2rot, ca_rot2euler
from o2r_pi2_controllers.managers import ManagerCasadiModel

In [ ]:
@configclass
class ModelCfg:
    name = "hunter"
    x_label = {'x' : 0,
                'y' : 1,
                'yaw' : 2,
                'front_wheel' : 3,
        }
    u_label = {'vx' : 0,
                'wfront_wheel' : 1,
        }
    length = 0.98
    width = 0.745
    height = 0.375
    wheels_distance_length = 0.65
    wheels_distance_width = 0.465
    wheels_radius = 0.33/2
    base2front = [0., 0., wheels_distance_length]
    T_base2front = np.vstack((np.hstack((np.identity(3),np.transpose(np.array([base2front])))),[0,0,0,1]))
    nq = 4
    nx = nq # state |x,y,orientation, arm angle n
    nu = 2 # contr |v,v_ang, v arm angle n  


You can then add methods that will be accessible thought the final model manager.

There is a required methods :`_T_base` and/or `_T_effector` witch takes x (the robot state), and returns a 4x4 matrix representing the pose of the base and/or the end effector. 

`_derivativ_sym` is a required method for the MPC solver (introduce in future examples). It takes the state x and the command u, and return the derivative x_dot.

In this example `T_base_front` and are not required to create the model manager, but will still be accessible through it.

In [ ]:
    def _T_base(self, x):
        R = ca_euler2rot([0., 0., x[2]])
        t = np.array([[x[0]], [x[1]], [0.]])
        T_base = vertcat(horzcat(R, t), np.array([[0., 0., 0., 1.]]))
        return T_base
    
    def _derivativ_sym(self, x, u): # one-steered model
        return vertcat(
        u[0] * cos(x[self.x_label['yaw']] + x[self.x_label['front_wheel']]),
        u[0] * sin(x[self.x_label['yaw']] + x[self.x_label['front_wheel']]),
        u[0] / self.wheels_distance_length * sin(x[self.x_label['front_wheel']]),
        u[1]
        )

    def T_base_front(self, x):
        T_base = self.T_base(x)
        world2frontaxle = ca.mtimes(T_base, self.T_base2front)
        return world2frontaxle


Now that the model configuration is ready, create the robot configuration. In future examples it will be possible to add it other types of configurations, as ros msgs or cost function.

In [ ]:
@configclass
class HunterCfg():
    robot_model = ModelCfg()

You get the whole robot config there:

In [ ]:
@configclass
class ModelCfg:
    name = "hunter"
    x_label = {'x' : 0,
                'y' : 1,
                'yaw' : 2,
                'front_wheel' : 3,
        }
    u_label = {'vx' : 0,
                'wfront_wheel' : 1,
        }
    length = 0.98
    width = 0.745
    height = 0.375
    wheels_distance_length = 0.65
    wheels_distance_width = 0.465
    wheels_radius = 0.33/2
    base2front = [0., 0., wheels_distance_length]
    T_base2front = np.vstack((np.hstack((np.identity(3),np.transpose(np.array([base2front])))),[0,0,0,1]))
    nq = 4
    nx = nq # state |x,y,orientation, arm angle n
    nu = 2 # contr |v,v_ang, v arm angle n  

    def _T_base(self, x):
        R = ca_euler2rot([0., 0., x[2]])
        t = np.array([[x[0]], [x[1]], [0.]])
        T_base = vertcat(horzcat(R, t), np.array([[0., 0., 0., 1.]]))
        return T_base
    
    def _derivativ_sym(self, x, u): # one-steered model
        return vertcat(
        u[0] * cos(x[self.x_label['yaw']] + x[self.x_label['front_wheel']]),
        u[0] * sin(x[self.x_label['yaw']] + x[self.x_label['front_wheel']]),
        u[0] / self.wheels_distance_length * sin(x[self.x_label['front_wheel']]),
        u[1]
        )

    def T_base_front(self, x):
        T_base = self.T_base(x)
        world2frontaxle = ca.mtimes(T_base, self.T_base2front)
        return world2frontaxle



@configclass
class HunterCfg():
    robot_model = ModelCfg()

___
## ManagerCasadiModel: model manager

Now that the robot configuration is ready, you can create the model manager:

In [ ]:
hunter = ManagerCasadiModel(HunterCfg())

This model manager allows you to call implemented methods as `init_state`, `T_base`, `T_effector` and `quat_effector`. But also custom ones that you have defined in ModelCfg (as T_base_front in our example).

In [ ]:
x = hunter.init_state()    
print(hunter.T_base(x))
print(hunter.T_effector(x))
print(hunter.quat_effector(x))
print(hunter.T_base_front(x))